# Лабораторная работа № 4
Есиков Сергей 

СПбАУ, 302 гр.

Вар 5

### Задание
![alt text](../tasks/4.png)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import t

In [ ]:
MY_VARIANT = 5
df = pd.read_csv('../data/4.csv', header=None, names=['variant', 'country', 'value'])

## 1. Выбрать данные своего варианта

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Загрузка данных (замените 'planets.csv' на имя вашего файла)
try:
    df = pd.read_csv("planets.csv")
except FileNotFoundError:
    # Игрушечные данные для демонстрации работы кода
    np.random.seed(42)
    planets_mock = ["Марс", "Венера"]
    mock_data = []
    for p, mu, sigma in zip(planets_mock, [3389, 6051], [10, 15]):
        for val in np.random.normal(mu, sigma, size=100):
            mock_data.append({"вариант": 1, "планета": p, "радиус": val})
    df = pd.DataFrame(mock_data)

# Приведем названия колонок к общему виду для удобства обращения
df.columns = ["variant", "planet", "radius"]

# Укажите ваш номер варианта, чтобы отфильтровать нужные строки
MY_VARIANT = 1
df_variant = df[df["variant"] == MY_VARIANT]

alpha = 0.05  # Вероятность 0.95 -> alpha = 0.05
results = []

print(f"--- Результаты анализа для варианта №{MY_VARIANT} ---")

# Группируем по планетам внутри выбранного варианта
for planet, group in df_variant.groupby("planet"):
    radii = group["radius"].values
    n = len(radii)

    if n < 2:
        print(
            f"Для планеты {planet} слишком мало данных ({n} изм.), интервал построить нельзя."
        )
        continue

    # Выборочные характеристики
    m = np.mean(radii)
    # Используем ddof=1 для получения несмещенной дисперсии S^2
    S = np.std(radii, ddof=1)

    # Квантиль распределения Стьюдента с (n-1) степенями свободы
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)

    # Вычисление половины ширины интервала (Margin of Error)
    margin = t_crit * S * np.sqrt(1 + 1 / n)

    # Границы интервала
    left_bound = m - margin
    right_bound = m + margin

    # Подсчет количества имеющихся измерений, попавших в этот интервал
    count_inside = np.sum((radii >= left_bound) & (radii <= right_bound))
    percentage = (count_inside / n) * 100

    print(f"\nПланета: {planet} (всего {n} измерений)")
    print(f"  Выборочное среднее (m): {m:.4f}")
    print(f"  Выборочное СКО (S): {S:.4f}")
    print(f"  Предиктивный интервал (95%): [{left_bound:.4f}; {right_bound:.4f}]")
    print(
        f"  Попало измерений из выборки в интервал: {count_inside} ({percentage:.2f}%)"
    )
